# 崩溃后继续执行：幂等、回放与补偿

> 状态：verified；本地教学账本；执行日期：2026-09-06。

这里的金额只是整数计数，不连接真实支付服务。先读[持久执行](../01-concepts/02-durable-execution.md)。观察重点是账本实际记录条数，不是程序是否打印“成功”。

执行说明：本次因环境禁止 Kernel socket，使用独立 Python 进程内 IPython 按单元顺序执行并保存真实输出；不是 Jupyter kernel 运行。普通 Jupyter 环境可按原顺序运行。

In [1]:
from pathlib import Path
import sys, json
from pprint import pprint
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "10-Knowledge").is_dir())
sys.path.insert(0, str(ROOT / "10-Knowledge/09-runtime-harness-environment/05-code/recoverable-runtime-python/src"))
from recoverable_runtime import EventStore, LocalLedger, Runner, InjectedCrash
import tempfile
naive_total = 0
for attempt in range(2):
    naive_total += 10
print("没有幂等键，两次尝试的效果：", naive_total)
assert naive_total == 20


没有幂等键，两次尝试的效果： 20


现在将本地事件与“外部服务”放在两个独立数据库，保留跨系统提交窗口。对每个故障位置关闭并重新打开数据库，然后用原 run_id/step_id 恢复。它模拟进程丢失内存的情况，不模拟损坏磁盘。

In [2]:
tmp = tempfile.TemporaryDirectory()
work = Path(tmp.name)
def open_runtime():
    store = EventStore(work / "events.sqlite")
    ledger = LocalLedger(work / "ledger.sqlite")
    return store, ledger, Runner(store, ledger)
store, ledger, runner = open_runtime()
rows, traces = [], {}
for point in ["before_effect", "after_effect", "after_checkpoint"]:
    before = ledger.snapshot()["charge_count"]
    try:
        runner.execute(point, "charge", 10, crash_at=point)
    except InjectedCrash:
        after_crash = ledger.snapshot()["charge_count"] - before
    store.close(); ledger.close()
    store, ledger, runner = open_runtime()
    first = runner.execute(point, "charge", 10)
    second = runner.execute(point, "charge", 10)
    assert first == second
    delta = ledger.snapshot()["charge_count"] - before
    assert delta == 1
    rows.append({"crash_at": point, "effect_count_at_crash": after_crash,
                 "effect_count_after_resume_and_repeat": delta,
                 "status": runner.replay(point)["charge"]["status"]})
    traces[point] = store.events(point)
pprint(rows)
pprint(ledger.snapshot())
assert ledger.snapshot()["net"] == 30


[{'crash_at': 'before_effect',
  'effect_count_after_resume_and_repeat': 1,
  'effect_count_at_crash': 0,
  'status': 'completed'},
 {'crash_at': 'after_effect',
  'effect_count_after_resume_and_repeat': 1,
  'effect_count_at_crash': 1,
  'status': 'completed'},
 {'crash_at': 'after_checkpoint',
  'effect_count_after_resume_and_repeat': 1,
  'effect_count_at_crash': 1,
  'status': 'completed'}]
{'charge_count': 3, 'charged': 30, 'net': 30, 'refund_count': 0, 'refunded': 0}


调用前崩溃，效果计数是 0；调用后和 Checkpoint 后崩溃，效果计数是 1。恢复时三种情况都只有一条逻辑效果。调用后崩溃最关键：本地没有完成记录，服务端却已执行；同幂等键让它返回原记录。

查看这个窗口的真实 trace。`attempt` 可以出现两次，`completed` 只代表已确认的结果，不能把尝试数当业务执行次数。

In [3]:
pprint(traces["after_effect"])
before_replay = ledger.snapshot()
state = runner.replay("after_effect")
runner.replay("after_effect")
after_replay = ledger.snapshot()
assert before_replay == after_replay
print("两次回放后账本未改变：", before_replay == after_replay)
pprint(state)
try:
    runner.execute("after_effect", "charge", 11)
except ValueError as exc:
    print("同 key 改金额被拒绝：", exc)
else:
    raise AssertionError("changed payload accepted")


[{'event': 'prepared',
  'payload': {'amount': 10},
  'seq': 5,
  'step_id': 'charge'},
 {'event': 'attempt', 'payload': {}, 'seq': 6, 'step_id': 'charge'},
 {'event': 'attempt', 'payload': {}, 'seq': 7, 'step_id': 'charge'},
 {'event': 'completed',
  'payload': {'amount': 10, 'key': '["after_effect","charge"]'},
  'seq': 8,
  'step_id': 'charge'}]
两次回放后账本未改变： True
{'charge': {'amount': 10,
            'result': {'amount': 10, 'key': '["after_effect","charge"]'},
            'status': 'completed'}}
同 key 改金额被拒绝： step key reused with different amount


接着补偿其中一条扣款：退款后、补偿状态保存前再次崩溃。恢复补偿也要去重，否则会退款两次。补偿是新业务动作，不删除历史扣款。

In [4]:
try:
    runner.compensate("after_effect", "charge", crash_after_refund=True)
except InjectedCrash:
    print("退款已发生，补偿状态未保存：", ledger.snapshot())
store.close(); ledger.close()
store, ledger, runner = open_runtime()
runner.compensate("after_effect", "charge")
runner.compensate("after_effect", "charge")
snapshot = ledger.snapshot()
pprint(snapshot)
assert snapshot["charge_count"] == 3 and snapshot["refund_count"] == 1
assert snapshot["net"] == 20
assert runner.replay("after_effect")["charge"]["status"] == "compensated"
evidence = {"data": "synthetic local ledger", "crashes": rows, "traces_before_compensation": traces,
            "compensated_trace": store.events("after_effect"), "final_ledger": snapshot,
            "replay_had_no_effect": before_replay == after_replay}
artifact = Path("artifacts/recovery.json")
artifact.parent.mkdir(exist_ok=True)
artifact.write_text(json.dumps(evidence, ensure_ascii=False, indent=2), encoding="utf-8")
store.close(); ledger.close(); tmp.cleanup()
print("结构化证据：", artifact)


退款已发生，补偿状态未保存： {'charged': 30, 'charge_count': 3, 'refunded': 10, 'refund_count': 1, 'net': 20}
{'charge_count': 3, 'charged': 30, 'net': 20, 'refund_count': 1, 'refunded': 10}
结构化证据： artifacts/recovery.json


结论限于本地单写者、支持原子幂等的模拟服务。无幂等键且无法查结果的真实外部服务，不会因为加入 Checkpoint 自动变安全。

练习：恢复时故意生成新的 step_id，观察为什么会新增记录；再查看[测试](../05-code/recoverable-runtime-python/tests/test_recovery.py)中的 pending 补偿拒绝。`execute` 会确保原动作完成，不是只读查询：若在调用前崩溃，执行它会新建扣款，因此不能把 `execute → compensate` 当成用户取消流程。